# Projet de synthèse — Analyse Sephora

Version corrigée et reproductible.

## Corrections principales

1. Conversion explicite des colonnes numériques avant toute analyse.
2. Nettoyage des valeurs catégorielles et validation des catégories anormales.
3. Calcul correct du taux de recommandation.
4. Export CSV robuste pour Power BI avec toutes les cellules entre guillemets.
5. Suppression des valeurs VADER écrites manuellement.
6. Modèle prédictif sans fuite de cible : `rating` est exclu du modèle principal.
7. Chemins regroupés dans une seule cellule de configuration.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# =========================
# À MODIFIER UNE SEULE FOIS
# =========================
PROJECT_DIR = Path(r"C:\uqo\s5\projet synthèse")
DATA_DIR = PROJECT_DIR / "dataset"
REVIEWS_DIR = DATA_DIR / "archive"

WEBSITE_FILE = (
    DATA_DIR
    / "sephora_website_dataset.csv"
    / "sephora_website_dataset.csv"
)

OUTPUT_DIR = PROJECT_DIR / "powerbi"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_FILES = [
    REVIEWS_DIR / "reviews_0-250.csv",
    REVIEWS_DIR / "reviews_250-500.csv",
    REVIEWS_DIR / "reviews_500-750.csv",
    REVIEWS_DIR / "reviews_750-1250.csv",
    REVIEWS_DIR / "reviews_1250-end.csv",
]

print("Dossier du projet :", PROJECT_DIR)
print("Dossier Power BI  :", OUTPUT_DIR)

## 1. Chargement sécurisé

Le notebook vérifie d'abord que chaque fichier existe. Cela évite de continuer avec un mauvais chemin ou un fichier manquant.

In [ ]:
missing = [p for p in [WEBSITE_FILE, *REVIEW_FILES] if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Fichiers introuvables :\n" + "\n".join(str(p) for p in missing)
    )

df_website = pd.read_csv(WEBSITE_FILE, low_memory=False)

review_parts = [
    pd.read_csv(path, low_memory=False)
    for path in REVIEW_FILES
]
df_reviews = pd.concat(review_parts, ignore_index=True)

print("Reviews :", df_reviews.shape)
print("Website :", df_website.shape)

## 2. Nettoyage des noms et suppression de la colonne technique

In [ ]:
df_reviews.columns = df_reviews.columns.str.strip()
df_website.columns = df_website.columns.str.strip()

df_reviews = df_reviews.drop(columns=["Unnamed: 0"], errors="ignore")

print("Colonnes Reviews :", df_reviews.columns.tolist())
print("Colonnes Website :", df_website.columns.tolist())

## 3. Conversion explicite des types

L'erreur principale venait du fait que Power BI pouvait recevoir certaines colonnes numériques comme texte.  
On transforme donc les colonnes critiques en nombres dans Python avant l'export.

In [ ]:
reviews_numeric = [
    "rating", "is_recommended", "helpfulness",
    "total_feedback_count", "total_neg_feedback_count",
    "total_pos_feedback_count", "price_usd"
]

website_numeric = [
    "rating", "number_of_reviews", "love", "price",
    "value_price", "online_only", "exclusive",
    "limited_edition", "limited_time_offer"
]

for col in reviews_numeric:
    if col in df_reviews.columns:
        df_reviews[col] = pd.to_numeric(df_reviews[col], errors="coerce")

for col in website_numeric:
    if col in df_website.columns:
        df_website[col] = pd.to_numeric(df_website[col], errors="coerce")

df_reviews["submission_time"] = pd.to_datetime(
    df_reviews["submission_time"],
    errors="coerce"
)

print(df_reviews[["rating", "is_recommended", "price_usd"]].dtypes)
print(df_website[["rating", "price", "number_of_reviews"]].dtypes)

## 4. Nettoyage des variables textuelles

Les espaces invisibles et les chaînes vides peuvent créer plusieurs catégories qui semblent identiques dans Power BI.

In [ ]:
reviews_text = ["skin_type", "brand_name", "product_name"]
website_text = ["brand", "category", "name"]

for col in reviews_text:
    if col in df_reviews.columns:
        df_reviews[col] = (
            df_reviews[col]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

for col in website_text:
    if col in df_website.columns:
        df_website[col] = (
            df_website[col]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

## 5. Contrôles de qualité

Ces tests doivent être exécutés avant l'analyse et avant chaque export vers Power BI.

In [ ]:
print("=== Valeurs manquantes — colonnes critiques ===")
print(df_reviews[[
    "rating", "is_recommended", "skin_type",
    "brand_name", "price_usd"
]].isna().sum())

print(df_website[[
    "rating", "price", "category",
    "number_of_reviews"
]].isna().sum())

print("\n=== Plages attendues ===")
print("Reviews rating :", df_reviews["rating"].min(), "à", df_reviews["rating"].max())
print("Website rating :", df_website["rating"].min(), "à", df_website["rating"].max())
print("is_recommended :", sorted(df_reviews["is_recommended"].dropna().unique()))

# Une catégorie ne devrait pas être uniquement un nombre comme 70.0.
category_text = df_website["category"].astype("string")
numeric_categories = category_text.str.fullmatch(r"\d+(?:[.,]\d+)?", na=False)

print("\nCatégories numériques suspectes :", int(numeric_categories.sum()))
if numeric_categories.any():
    print(df_website.loc[
        numeric_categories,
        ["brand", "category", "name", "price", "rating"]
    ].head(20))

# On retire uniquement les lignes clairement invalides pour les graphiques par catégorie.
df_website = df_website.loc[~numeric_categories].copy()

## 6. Doublons

On affiche les doublons au lieu d'affirmer automatiquement qu'il n'y en a aucun.

In [ ]:
print("Doublons Reviews :", df_reviews.duplicated().sum())
print("Doublons Website :", df_website.duplicated().sum())

# Suppression prudente des doublons strictement identiques.
df_reviews = df_reviews.drop_duplicates().copy()
df_website = df_website.drop_duplicates().copy()

## 7. Variables dérivées

In [ ]:
df_reviews["annee"] = df_reviews["submission_time"].dt.year.astype("Int64")
df_reviews["mois"] = df_reviews["submission_time"].dt.month.astype("Int64")

df_reviews["sentiment_label"] = pd.cut(
    df_reviews["rating"],
    bins=[-np.inf, 2, 3, np.inf],
    labels=["négatif", "neutre", "positif"]
).astype("string")

df_website["gamme_prix"] = pd.cut(
    df_website["price"],
    bins=[-np.inf, 25, 75, np.inf],
    labels=["entrée de gamme", "milieu de gamme", "haut de gamme"],
    ordered=True
).astype("string")

df_reviews["gamme_prix_reviews"] = pd.cut(
    df_reviews["price_usd"],
    bins=[-np.inf, 25, 75, np.inf],
    labels=["entrée de gamme", "milieu de gamme", "haut de gamme"],
    ordered=True
).astype("string")

## 8. Résumé de validation

In [ ]:
print("=== Reviews ===")
print(df_reviews["rating"].describe())
print("\nTaux global de recommandation :",
      df_reviews["is_recommended"].mean())

print("\nTaux par type de peau :")
print(
    df_reviews.groupby("skin_type", observed=True)["is_recommended"]
    .agg(taux_recommandation="mean", nombre_avis="count")
    .sort_values("taux_recommandation", ascending=False)
)

print("\n=== Website ===")
print(df_website[["price", "rating", "number_of_reviews"]].describe())

print("\nNote moyenne par gamme de prix :")
print(
    df_website.groupby("gamme_prix", observed=True)["rating"]
    .agg(note_moyenne="mean", nombre_produits="count")
)

## 9. Analyse exploratoire

In [ ]:
rating_counts = df_reviews["rating"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(rating_counts.index.astype(str), rating_counts.values)
plt.title("Distribution des notes clients")
plt.xlabel("Note")
plt.ylabel("Nombre d'avis")
plt.tight_layout()
plt.show()

In [ ]:
top_brands = (
    df_reviews.groupby("brand_name", observed=True)
    .agg(nombre_avis=("rating", "count"),
         note_moyenne=("rating", "mean"))
    .sort_values("nombre_avis", ascending=False)
    .head(10)
)

print(top_brands)

plt.figure(figsize=(10, 5))
plt.bar(top_brands.index, top_brands["nombre_avis"])
plt.title("Top 10 marques par volume d'avis")
plt.xlabel("Marque")
plt.ylabel("Nombre d'avis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
note_skin = (
    df_reviews.groupby("skin_type", observed=True)
    .agg(note_moyenne=("rating", "mean"),
         taux_recommandation=("is_recommended", "mean"),
         nombre_avis=("rating", "count"))
    .sort_values("note_moyenne", ascending=False)
)

print(note_skin)

In [ ]:
price_rating = df_website[["price", "rating"]].dropna()
correlation = price_rating["price"].corr(price_rating["rating"])
print(f"Corrélation prix / note : {correlation:.4f}")

price_band_summary = (
    df_website.groupby("gamme_prix", observed=True)
    .agg(note_moyenne=("rating", "mean"),
         nombre_produits=("rating", "count"))
)
print(price_band_summary)

In [ ]:
category_summary = (
    df_website.dropna(subset=["category", "rating"])
    .groupby("category", observed=True)
    .agg(note_moyenne=("rating", "mean"),
         nombre_produits=("rating", "count"),
         total_avis=("number_of_reviews", "sum"))
)

# Un minimum de produits évite de classer en tête une catégorie représentée par un seul produit.
category_top = (
    category_summary[category_summary["nombre_produits"] >= 5]
    .sort_values("note_moyenne", ascending=False)
    .head(15)
)

print(category_top)

## 10. Analyse VADER

Le graphique est maintenant calculé à partir des résultats réels.  
Les pourcentages ne sont plus écrits manuellement.

In [ ]:
# Décommenter une seule fois si nécessaire :
# %pip install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

sample = (
    df_reviews["review_text"]
    .dropna()
    .astype(str)
    .sample(n=min(10_000, df_reviews["review_text"].notna().sum()),
            random_state=42)
)

compound_scores = sample.map(
    lambda text: analyzer.polarity_scores(text)["compound"]
)

vader_labels = pd.cut(
    compound_scores,
    bins=[-np.inf, -0.05, 0.05, np.inf],
    labels=["négatif", "neutre", "positif"],
    include_lowest=True
)

vader_distribution = (
    vader_labels.value_counts(normalize=True)
    .reindex(["positif", "négatif", "neutre"])
    .fillna(0)
    .mul(100)
)

print(vader_distribution.round(1))

plt.figure(figsize=(8, 5))
plt.bar(vader_distribution.index, vader_distribution.values)
plt.title("Distribution des sentiments — VADER")
plt.xlabel("Sentiment")
plt.ylabel("Pourcentage")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

## 11. Modèle prédictif sans fuite de cible

La version précédente utilisait `rating` pour prédire `is_recommended`.  
Ces deux variables décrivent presque la même opinion, ce qui produisait une précision artificiellement élevée.

Le modèle principal ci-dessous exclut donc `rating`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

model_data = df_reviews[[
    "skin_type", "price_usd", "brand_name", "is_recommended"
]].dropna(subset=["is_recommended"]).copy()

model_data = model_data[
    model_data["is_recommended"].isin([0, 1])
].copy()

X = model_data[["skin_type", "price_usd", "brand_name"]]
y = model_data["is_recommended"].astype(int)

categorical_features = ["skin_type", "brand_name"]
numeric_features = ["price_usd"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=100
                ))
            ]),
            categorical_features
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        )
    ]
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=500,
        class_weight="balanced",
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

print(classification_report(y_test, predictions, digits=3))
print(
    "Balanced accuracy :",
    round(balanced_accuracy_score(y_test, predictions), 3)
)

## 12. Export propre pour Power BI

`QUOTE_ALL` protège les cellules contenant des virgules, des guillemets ou des retours à la ligne.  
Les colonnes numériques sont déjà converties avant l'export.

In [ ]:
reviews_columns = [
    "rating", "is_recommended", "skin_type",
    "brand_name", "price_usd", "annee", "mois",
    "sentiment_label", "product_name"
]

website_columns = [
    "brand", "category", "name", "price",
    "rating", "number_of_reviews",
    "online_only", "exclusive",
    "limited_edition", "gamme_prix"
]

df_reviews_export = df_reviews[reviews_columns].copy()
df_website_export = df_website[website_columns].copy()

# Validation finale
assert df_reviews_export["rating"].between(1, 5).all()
assert set(df_reviews_export["is_recommended"].dropna().unique()).issubset({0, 1})
assert df_website_export["rating"].dropna().between(0, 5).all()
assert not (
    df_website_export["category"]
    .astype("string")
    .str.fullmatch(r"\d+(?:[.,]\d+)?", na=False)
).any()

reviews_csv = OUTPUT_DIR / "reviews_clean.csv"
website_csv = OUTPUT_DIR / "website_clean.csv"

df_reviews_export.to_csv(
    reviews_csv,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n"
)

df_website_export.to_csv(
    website_csv,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n"
)

print("Exports créés :")
print(reviews_csv)
print(website_csv)
print("Reviews export :", df_reviews_export.shape)
print("Website export :", df_website_export.shape)

## 13. Contrôle de réimportation

Cette étape relit exactement les fichiers qui seront utilisés par Power BI.

In [ ]:
reviews_check = pd.read_csv(reviews_csv, low_memory=False)
website_check = pd.read_csv(website_csv, low_memory=False)

print("Reviews relu :", reviews_check.shape)
print("Website relu :", website_check.shape)

print("\nTypes Reviews :")
print(reviews_check.dtypes)

print("\nTypes Website :")
print(website_check.dtypes)

print("\nMoyenne website.rating :", website_check["rating"].mean())
print("Taux global de recommandation :",
      reviews_check["is_recommended"].mean())

print("\nTaux de recommandation par type de peau :")
print(
    reviews_check.groupby("skin_type")["is_recommended"]
    .mean()
    .sort_values(ascending=False)
)